# VisionDecode — Final Model (Weighted Ensemble)

The final submission combines the three trained classification models by **weighted soft-voting**
(averaging softmax probabilities per character position, then argmax):

| Model | Backbone + head | Checkpoint | Weight |
|-------|-----------------|------------|--------|
| Model 1 | EfficientNet-B0 + 6 heads | `model1_efficientnet.pth` | 0.25 |
| Model 2 | EfficientNet-B0 + BiGRU (CRNN) | `model2_crnn_bigru.pth` | 0.25 |
| Model 3 | MobileNetV3-Small + GRU | `model3_mobilenet_gru.pth` | 0.50 |

All three output a `(B, 6, 33)` grid over the 33-char alphabet (`0-9A-Z` minus I/L/O) and share
the same `100×200` ImageNet transform, so their per-position probabilities can be averaged
directly. This notebook is **self-contained**: it redefines the helpers and model classes, loads
the checkpoints, reports the ensemble's validation CER, and writes the final submission CSV.

> Requires the three checkpoints in `../Artifacts/`. Train them first via
> `prototype_model.ipynb` if they don't exist yet.

In [1]:
import os
import glob
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
print('Using device:', device)

Using device: cuda


## Shared helpers (charset, CER, dataset, loaders)

In [2]:
# --- Charset (33 classes; no I/L/O) ---
CHARS = '0123456789ABCDEFGHJKMNPQRSTUVWXYZ'
NUM_CLASSES = len(CHARS)        # 33
CODE_LEN = 6

char_to_idx = {c: i for i, c in enumerate(CHARS)}
idx_to_char = {i: c for i, c in enumerate(CHARS)}

def encode(text):
    return torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)

def decode(indices):
    return ''.join(idx_to_char[int(i)] for i in indices)

# --- CER via Levenshtein ---
def levenshtein(a, b):
    n, m = len(a), len(b)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, m + 1):
            cur = dp[j]
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + (a[i - 1] != b[j - 1]))
            prev = cur
    return dp[m]

def cer(preds, targets):
    edits = sum(levenshtein(p, t) for p, t in zip(preds, targets))
    chars = sum(len(t) for t in targets)
    return edits / max(chars, 1)

In [3]:
# --- Dataset + loader factory (same seed/split as training) ---
DATA_DIR = '../data'

class CaptchaDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        df = pd.read_csv(csv_file)
        df = df[df['text'].astype(str).str.len() == CODE_LEN].reset_index(drop=True)
        df = df[df['text'].apply(lambda t: all(c in char_to_idx for c in str(t)))].reset_index(drop=True)
        self.df, self.img_dir, self.transform = df, img_dir, transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row['image'])).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, encode(row['text'])

def make_loaders(transform, batch_size=64, val_frac=0.1, seed=42):
    full = CaptchaDataset(os.path.join(DATA_DIR, 'train-labels.csv'),
                          os.path.join(DATA_DIR, 'train_images'), transform)
    val_size = int(val_frac * len(full))
    train, val = random_split(full, [len(full) - val_size, val_size],
                              generator=torch.Generator().manual_seed(seed))
    tl = DataLoader(train, batch_size=batch_size, shuffle=True,  num_workers=0)
    vl = DataLoader(val,   batch_size=128,        shuffle=False, num_workers=0)
    return tl, vl

# Shared transform for all three models (100x200, ImageNet norm).
imagenet_tf = transforms.Compose([
    transforms.Resize((100, 200)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## Training PipeLine

In [6]:
def decode_logits(logits):
    idx = logits.argmax(dim=2).cpu()
    return [decode(row) for row in idx]

@torch.no_grad()
def evaluate_cls(model, loader):
    model.eval()
    preds, trues, seq_ok, seq_n = [], [], 0, 0
    for images, labels in loader:
        out = model(images.to(device))            # (B, 6, 33)
        p = out.argmax(dim=2).cpu()
        for pi, ti in zip(p, labels):
            preds.append(decode(pi)); trues.append(decode(ti))
        seq_ok += (p == labels).all(dim=1).sum().item()
        seq_n  += labels.size(0)
    return cer(preds, trues), seq_ok / max(seq_n, 1)

def cls_loss(logits, labels):
    return sum(F.cross_entropy(logits[:, p, :], labels[:, p]) for p in range(CODE_LEN))

def train_classifier(model, train_loader, val_loader, epochs=15, lr=1e-3, ckpt=None):
    model = model.to(device)
    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=2)
    best = float('inf')
    if ckpt:
        os.makedirs(os.path.dirname(ckpt), exist_ok=True)
    for epoch in range(1, epochs + 1):
        model.train(); running = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            opt.zero_grad()
            loss = cls_loss(model(images), labels)
            loss.backward(); opt.step()
            running += loss.item() * images.size(0)
        tr = running / len(train_loader.dataset)
        vcer, sacc = evaluate_cls(model, val_loader); sched.step(vcer)
        print(f'Epoch {epoch:2d} | loss {tr:.4f} | val CER {vcer:.4f} | full-code acc {sacc:.4f}')
        if ckpt and vcer < best:
            best = vcer; torch.save(model.state_dict(), ckpt)
            print(f'  ↳ best CER {best:.4f} saved -> {ckpt}')
    return best

## Model definitions

Identical to the architectures trained in `prototype_model.ipynb` — they must match exactly for
`load_state_dict` to succeed.

In [14]:
class CaptchaEfficientNet_Advanced(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN):
        super().__init__()
        self.code_len = code_len
        self.model = models.efficientnet_b0(weights='DEFAULT')
        for p in self.model.parameters():
            p.requires_grad = False

        for p in self.model.features[-4:].parameters():
            p.requires_grad = True
        in_features = self.model.classifier[1].in_features
        
        self.classifiers = nn.ModuleList([
            nn.Sequential(
                nn.Dropout(0.3), 
                nn.Linear(in_features, 256),
                nn.BatchNorm1d(256),
                nn.ReLU(),
                nn.Dropout(0.3), 
                nn.Linear(256, 512),  
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(512, num_classes)
            )
            for _ in range(code_len)
        ])

    def forward(self, x):
        f = self.model.features(x)
        f = torch.flatten(self.model.avgpool(f), 1)
        return torch.stack([h(f) for h in self.classifiers], dim=1)


class CRNN_EfficientNet_Advanced(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN, hidden=320):  # 256→320
        super().__init__()
        self.code_len = code_len
        backbone = models.efficientnet_b0(weights='DEFAULT')
        for p in backbone.parameters():
            p.requires_grad = False
        # Unfreeze more layers
        for p in backbone.features[-4:].parameters():  
            p.requires_grad = True
        self.features = backbone.features
        self.pool = nn.AdaptiveAvgPool2d((1, code_len))
        
        # Increased hidden size and added 3rd layer
        self.gru = nn.GRU(1280, hidden, num_layers=3, batch_first=True,  
                          bidirectional=True, dropout=0.25)  
        self.layer_norm = nn.LayerNorm(hidden * 2)  
        self.fc = nn.Sequential(
            nn.Linear(hidden * 2, 512), 
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        f = self.features(x)
        f = self.pool(f).squeeze(2).permute(0, 2, 1)
        seq, _ = self.gru(f)
        seq = self.layer_norm(seq)
        return self.fc(seq)

class CRNN_MobileNet_Advanced(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN, hidden=128):
        super().__init__()
        self.code_len = code_len
        backbone = models.mobilenet_v3_small(weights='DEFAULT')
        self.features = backbone.features          # output channels: 576
        feat_ch = 576
        self.pool = nn.AdaptiveAvgPool2d((1, code_len))
        self.gru = nn.GRU(feat_ch, hidden, num_layers=4, batch_first=True,
                          bidirectional=True, dropout=0.3)
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, x):
        f = self.features(x)                       # (B, 576, H, W)
        f = self.pool(f).squeeze(2).permute(0, 2, 1)  # (B, 6, 576)
        seq, _ = self.gru(f)
        return self.fc(seq)                        # (B, 6, 33)


In [15]:
model1 = CaptchaEfficientNet_Advanced().to(device)
model2 = CRNN_EfficientNet_Advanced().to(device)
model3 = CRNN_MobileNet_Advanced().to(device)

print('Model 1 params:', sum(p.numel() for p in model1.parameters() if p.requires_grad))
print('Model 2 params:', sum(p.numel() for p in model2.parameters() if p.requires_grad))
print('Model 3 params:', sum(p.numel() for p in model3.parameters() if p.requires_grad))

Model 1 params: 6560654
Model 2 params: 10815209
Model 3 params: 2367041


In [8]:
train_loader, val_loader = make_loaders(imagenet_tf)
train_classifier(model1, train_loader, val_loader, epochs=20, lr=1e-3,
                 ckpt='../Artifacts/FinalModel_model1_efficientnet.pth')

Epoch  1 | loss 13.2489 | val CER 0.2797 | full-code acc 0.1261
  ↳ best CER 0.2797 saved -> ../Artifacts/FinalModel_model1_efficientnet.pth
Epoch  2 | loss 4.0395 | val CER 0.1047 | full-code acc 0.5093
  ↳ best CER 0.1047 saved -> ../Artifacts/FinalModel_model1_efficientnet.pth
Epoch  3 | loss 2.1122 | val CER 0.0700 | full-code acc 0.6353
  ↳ best CER 0.0700 saved -> ../Artifacts/FinalModel_model1_efficientnet.pth
Epoch  4 | loss 1.4153 | val CER 0.0456 | full-code acc 0.7554
  ↳ best CER 0.0456 saved -> ../Artifacts/FinalModel_model1_efficientnet.pth
Epoch  5 | loss 1.0917 | val CER 0.0405 | full-code acc 0.7794
  ↳ best CER 0.0405 saved -> ../Artifacts/FinalModel_model1_efficientnet.pth
Epoch  6 | loss 0.8815 | val CER 0.0313 | full-code acc 0.8214
  ↳ best CER 0.0313 saved -> ../Artifacts/FinalModel_model1_efficientnet.pth
Epoch  7 | loss 0.7314 | val CER 0.0303 | full-code acc 0.8329
  ↳ best CER 0.0303 saved -> ../Artifacts/FinalModel_model1_efficientnet.pth
Epoch  8 | loss 0.6

0.012756378189094548

In [9]:
train_classifier(model2, train_loader, val_loader, epochs=25, lr=1e-3,
                 ckpt='../Artifacts/FinalModel_model2_crnn_gru.pth')


Epoch  1 | loss 8.1728 | val CER 0.1278 | full-code acc 0.4492
  ↳ best CER 0.1278 saved -> ../Artifacts/FinalModel_model2_crnn_gru.pth
Epoch  2 | loss 1.9475 | val CER 0.0640 | full-code acc 0.6688
  ↳ best CER 0.0640 saved -> ../Artifacts/FinalModel_model2_crnn_gru.pth
Epoch  3 | loss 1.1201 | val CER 0.0455 | full-code acc 0.7534
  ↳ best CER 0.0455 saved -> ../Artifacts/FinalModel_model2_crnn_gru.pth
Epoch  4 | loss 0.8291 | val CER 0.0429 | full-code acc 0.7679
  ↳ best CER 0.0429 saved -> ../Artifacts/FinalModel_model2_crnn_gru.pth
Epoch  5 | loss 0.6894 | val CER 0.0380 | full-code acc 0.7934
  ↳ best CER 0.0380 saved -> ../Artifacts/FinalModel_model2_crnn_gru.pth
Epoch  6 | loss 0.5766 | val CER 0.0358 | full-code acc 0.8054
  ↳ best CER 0.0358 saved -> ../Artifacts/FinalModel_model2_crnn_gru.pth
Epoch  7 | loss 0.5262 | val CER 0.0322 | full-code acc 0.8224
  ↳ best CER 0.0322 saved -> ../Artifacts/FinalModel_model2_crnn_gru.pth
Epoch  8 | loss 0.4590 | val CER 0.0303 | full-c

0.014090378522594631

In [16]:
train_classifier(model3, train_loader, val_loader, epochs=25, lr=1e-3,
                 ckpt='../Artifacts/FinalModel_model3_mobilenet_gru.pth')

Epoch  1 | loss 9.8779 | val CER 0.4184 | full-code acc 0.0450
  ↳ best CER 0.4184 saved -> ../Artifacts/FinalModel_model3_mobilenet_gru.pth
Epoch  2 | loss 2.5562 | val CER 0.1143 | full-code acc 0.4887
  ↳ best CER 0.1143 saved -> ../Artifacts/FinalModel_model3_mobilenet_gru.pth
Epoch  3 | loss 1.3083 | val CER 0.0669 | full-code acc 0.6638
  ↳ best CER 0.0669 saved -> ../Artifacts/FinalModel_model3_mobilenet_gru.pth
Epoch  4 | loss 0.7989 | val CER 0.0661 | full-code acc 0.6548
  ↳ best CER 0.0661 saved -> ../Artifacts/FinalModel_model3_mobilenet_gru.pth
Epoch  5 | loss 0.6075 | val CER 0.0392 | full-code acc 0.7859
  ↳ best CER 0.0392 saved -> ../Artifacts/FinalModel_model3_mobilenet_gru.pth
Epoch  6 | loss 0.4879 | val CER 0.0434 | full-code acc 0.7689
Epoch  7 | loss 0.3956 | val CER 0.0394 | full-code acc 0.7824
Epoch  8 | loss 0.3397 | val CER 0.0272 | full-code acc 0.8519
  ↳ best CER 0.0272 saved -> ../Artifacts/FinalModel_model3_mobilenet_gru.pth
Epoch  9 | loss 0.2976 | val

0.011422377855594464

## Ensemble — validation CER

Averages weighted softmax probabilities across the models position-wise, then argmax. Because all
models use the same transform and `make_loaders` uses a fixed seed, the per-model validation
loaders iterate the same samples in the same order, so labels stay aligned.

In [17]:
@torch.no_grad()
def ensemble_predict(models_with_transforms, loader_builder, weights=None):
    n = len(models_with_transforms)
    weights = weights or [1.0] * n
    # Build a loader per transform; assumes identical split (same seed) so labels align.
    loaders = [loader_builder(tf)[1] for _, tf in models_with_transforms]  # val loaders
    for m, _ in models_with_transforms:
        m.eval().to(device)

    preds, trues = [], []
    iters = [iter(l) for l in loaders]
    while True:
        try:
            batches = [next(it) for it in iters]
        except StopIteration:
            break
        labels = batches[0][1]
        prob_sum = 0.0
        for w, (m, _), (images, _) in zip(weights, models_with_transforms, batches):
            prob_sum = prob_sum + w * F.softmax(m(images.to(device)), dim=2).cpu()
        idx = prob_sum.argmax(dim=2)
        for pi, ti in zip(idx, labels):
            preds.append(decode(pi)); trues.append(decode(ti))
    seq_acc = np.mean([p == t for p, t in zip(preds, trues)])
    return cer(preds, trues), seq_acc, preds

In [18]:
# --- Load the three trained models ---
ART = '../Artifacts'

model1 = CaptchaEfficientNet_Advanced().to(device)
model1.load_state_dict(torch.load(f'{ART}/FinalModel_model1_efficientnet.pth', map_location=device))

model2 = CRNN_EfficientNet_Advanced().to(device)
model2.load_state_dict(torch.load(f'{ART}/FinalModel_model2_crnn_gru.pth', map_location=device))

model3 = CRNN_MobileNet_Advanced().to(device)
model3.load_state_dict(torch.load(f'{ART}/FinalModel_model3_mobilenet_gru.pth', map_location=device))

# (model, transform) pairs and ensemble weights
combo = [(model1, imagenet_tf), (model2, imagenet_tf), (model3, imagenet_tf)]
WEIGHTS = [0.25, 0.25, 0.50]

e_cer, e_acc, _ = ensemble_predict(combo, make_loaders, weights=WEIGHTS)
print(f'Ensemble  | val CER {e_cer:.4f} | full-code acc {e_acc:.4f}')

Ensemble  | val CER 0.0040 | full-code acc 0.9760


## Generate the final submission CSV

Runs the same weighted soft-vote over every image in `../data/test_images` and writes the file in
the required `image,prediction` format. **Replace `<name>` / `<enroll>` before submitting.**

In [19]:
class TestDataset(Dataset):
    def __init__(self, img_dir, transform):
        self.paths = sorted(glob.glob(os.path.join(img_dir, '*.png')))
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        p = self.paths[idx]
        return self.transform(Image.open(p).convert('RGB')), os.path.basename(p)

def _test_collate(batch):
    imgs, names = zip(*batch)
    return torch.stack(imgs, 0), list(names)

@torch.no_grad()
def ensemble_submit(models_with_transforms, test_dir, weights=None,
                    out_path='../submission_<name>_<enroll>.csv', batch_size=128):
    n = len(models_with_transforms)
    weights = weights or [1.0] * n
    # One loader per transform; sorted paths => identical ordering across models.
    loaders = [DataLoader(TestDataset(test_dir, tf), batch_size=batch_size,
                          shuffle=False, num_workers=0, collate_fn=_test_collate)
               for _, tf in models_with_transforms]
    for m, _ in models_with_transforms:
        m.eval().to(device)

    rows = []
    iters = [iter(l) for l in loaders]
    while True:
        try:
            batches = [next(it) for it in iters]
        except StopIteration:
            break
        names = batches[0][1]
        prob_sum = 0.0
        for w, (m, _), (images, _) in zip(weights, models_with_transforms, batches):
            prob_sum = prob_sum + w * F.softmax(m(images.to(device)), dim=2).cpu()
        idx = prob_sum.argmax(dim=2)
        for name, pi in zip(names, idx):
            rows.append((name, decode(pi)))

    sub = pd.DataFrame(rows, columns=['image', 'prediction'])
    sub.to_csv(out_path, index=False)
    print(f'Wrote {len(sub)} predictions -> {out_path}')
    return sub

submission = ensemble_submit(combo, os.path.join(DATA_DIR, 'test_images'),
                             weights=WEIGHTS,
                             out_path='../Result/submission_AryanSharma_24113024.csv')
submission.head(10)

Wrote 5000 predictions -> ../Result/submission_AryanSharma_24113024.csv


,image,prediction
0,test-0.png,QVTQ8A
1,test-1.png,7PSW9D
2,test-10.png,7DUP98
3,test-100.png,75Z4WT
4,test-1000.png,QAKZ7V
5,test-1001.png,R6MERY
6,test-1002.png,CHXX67
7,test-1003.png,9NV2WP
8,test-1004.png,F58TDZ
9,test-1005.png,FFTFRX


In [63]:
df = pd.read_csv('../Result/submission_AryanSharma_24113024.csv')
df["index"] = df['image'].str.split('-').str[1].str.split('.').str[0]
df["index"] = df["index"].astype(int)
df = df.sort_values('index').reset_index(drop=True)
df = df.drop(columns=['index'])
df.to_csv('../Result/submission_AryanSharma_24113024.csv', index=False)